In [1]:
!pip install -q accelerate -U
!pip install -q bitsandbytes -U
!pip install -q trl -U
!pip install -q peft -U
!pip install -q transformers -U
!pip install -q datasets -U
!pip install torchinfo

In [1]:
from torchinfo import summary
import os
import pandas as pd
import torch
from datasets import load_dataset, Dataset, DatasetDict
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from trl.extras.dataset_formatting import FORMAT_MAPPING, instructions_formatting_function, conversations_formatting_function

In [2]:
repo_id = 'microsoft/Phi-3-mini-4k-instruct'
tokenizer_1 = AutoTokenizer.from_pretrained(repo_id)

base_model_id = "microsoft/phi-2"
model = AutoModelForCausalLM.from_pretrained(base_model_id, trust_remote_code=True,
                                             torch_dtype=torch.float16, load_in_8bit=True)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
tokenizer_1

LlamaTokenizerFast(name_or_path='microsoft/Phi-3-mini-4k-instruct', vocab_size=32000, model_max_length=4096, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=False),
	32000: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<|placeholder1|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=Tr

In [4]:
tokenizer

CodeGenTokenizerFast(name_or_path='microsoft/phi-2', vocab_size=50257, model_max_length=2048, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50257: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50258: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50259: AddedToken("                             ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50260: AddedToken("                            ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50261: AddedToken("              

In [5]:
print(tokenizer.chat_template)
print(tokenizer_1.chat_template)

None
{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [6]:
tokenizer.chat_template = tokenizer_1.chat_template
print(tokenizer.chat_template)

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [7]:
tokenizer.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>'}

In [8]:
special_tokens_dict = {'unk_token': '<unk>',
                       'eos_token': '<|endoftext|>',
                       'bos_token': '<s>',
                       'pad_token': '<pad>'
                       }
special_tokens_dict

{'unk_token': '<unk>',
 'eos_token': '<|endoftext|>',
 'bos_token': '<s>',
 'pad_token': '<pad>'}

In [9]:
tokenizer.add_special_tokens(special_tokens_dict)
tokenizer

CodeGenTokenizerFast(name_or_path='microsoft/phi-2', vocab_size=50257, model_max_length=2048, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50257: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50258: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50259: AddedToken("                             ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50260: AddedToken("                            ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50261: AddedToken("          

In [10]:
new_toks  = ['<|system|>', '<|user|>', '<|assistant|>', '<|end|>']
new_toks

['<|system|>', '<|user|>', '<|assistant|>', '<|end|>']

In [11]:
tokenizer.add_tokens(new_toks)
tokenizer

CodeGenTokenizerFast(name_or_path='microsoft/phi-2', vocab_size=50257, model_max_length=2048, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50257: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50258: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50259: AddedToken("                             ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50260: AddedToken("                            ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50261: AddedToken("          

In [12]:
model.model.embed_tokens

Embedding(51200, 2560)

In [13]:
tokenizer.padding_side='left'
tokenizer.padding_side

'left'

In [14]:
print(model.get_memory_footprint()/1e6)

3042.785344


In [15]:
model

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
          (dense): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear8bitLt(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear8bitLt(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_

In [16]:
model = prepare_model_for_kbit_training(model)
target_modules = ["q_proj", "k_proj", "v_proj", "dense",  "fc1", "fc2"]
config = LoraConfig(
    # the rank of the adapter, the lower the fewer parameters you'll need to train
    r=16,
    lora_alpha=32, # multiplier, usually 2*r
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    # Newer models, such as Phi-3 at time of writing, may require
    # manually setting target modules
    target_modules=target_modules,
    )

model = get_peft_model(model, config)
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PhiForCausalLM(
      (model): PhiModel(
        (embed_tokens): Embedding(51200, 2560)
        (layers): ModuleList(
          (0-31): 32 x PhiDecoderLayer(
            (self_attn): PhiAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2560, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Line

In [17]:
print(model.get_memory_footprint()/1e6)

3663.360064


In [18]:
train_p, tot_p = model.get_nb_trainable_parameters()
print(f'Trainable parameters:      {train_p/1e6:.2f}M')
print(f'Total parameters:          {tot_p/1e6:.2f}M')
print(f'% of trainable parameters: {100*train_p/tot_p:.2f}%')

Trainable parameters:      23.59M
Total parameters:          2803.28M
% of trainable parameters: 0.84%


In [19]:
summary(model)

Layer (type:depth-idx)                                            Param #
PeftModelForCausalLM                                              --
├─LoraModel: 1-1                                                  --
│    └─PhiForCausalLM: 2-1                                        --
│    │    └─PhiModel: 3-1                                         2,672,153,600
│    │    └─Linear: 3-2                                           (131,123,200)
Total params: 2,803,276,800
Trainable params: 23,592,960
Non-trainable params: 2,779,683,840

In [20]:
ds_1 = load_dataset("guanaco/guanaco")

In [21]:
ds_1

DatasetDict({
    train: Dataset({
        features: ['text', 'prompt', 'response'],
        num_rows: 48701
    })
})

In [22]:
ds_1['train'][0]

{'text': "Describe the following:\nA tall, thin man with dark hair styled in a slick pompadour.### Response:\nThe man is quite tall with a slender physique. His dark hair is styled in a sleek, classic pompadour with a slight sheen to it, suggesting perhaps he spends time grooming it. The hair appears to be well-maintained, as though he's put some effort into it. The dark color of his hair contrasts with his pale skin, which gives him a somewhat striking appearance. He conveys a sense of poise and confidence, as though he's comfortable in his own skin. Overall, he seems to exude a sophisticated, polished vibe.",
 'prompt': 'Describe the following:\nA tall, thin man with dark hair styled in a slick pompadour.',
 'response': "The man is quite tall with a slender physique. His dark hair is styled in a sleek, classic pompadour with a slight sheen to it, suggesting perhaps he spends time grooming it. The hair appears to be well-maintained, as though he's put some effort into it. The dark col

In [23]:
def conv_chat_format(example):
    messages = []
    for k in example.keys():
        if k == 'prompt':
            role = 'user'
            content = example[k]
            user_dict = {'role': role, 'content': content}
        elif k == 'response':
            role = 'assistant'
            content = example[k]
            asst_dict = {'role': role, 'content': content}
        else:
            pass
    messages.append(user_dict)
    messages.append(asst_dict)
    return {'messages': messages}

In [24]:
ds_1 = ds_1.map(conv_chat_format, batched=False)

In [25]:
ds_1

DatasetDict({
    train: Dataset({
        features: ['text', 'prompt', 'response', 'messages'],
        num_rows: 48701
    })
})

In [26]:
ds_1['train'][1010]

{'text': "Describe the following:\nA carefree and laid-back surfer### Response:\nA carefree and laid-back surfer is a person who loves to ride the waves and embrace the thrill of the ocean. This type of surfer doesn't worry too much about anything and takes life as it comes, enjoying every moment along the way. They are easygoing and relaxed, and their positive energy radiates from their approach to life. They may spend all day surfing, socializing with other surfers, and taking in the beautiful surroundings. This type of surfer isn't concerned with material possessions and prioritizes living in the present moment, cherishing the simple pleasures in life.",
 'prompt': 'Describe the following:\nA carefree and laid-back surfer',
 'response': "A carefree and laid-back surfer is a person who loves to ride the waves and embrace the thrill of the ocean. This type of surfer doesn't worry too much about anything and takes life as it comes, enjoying every moment along the way. They are easygoin

In [27]:
df = ds_1['train'].to_pandas()

In [28]:
df

,text,prompt,response,messages
0,"Describe the following:\nA tall, thin man with...","Describe the following:\nA tall, thin man with...",The man is quite tall with a slender physique....,"[{'content': 'Describe the following: A tall, ..."
1,"Describe the following:\nA short, plump woman ...","Describe the following:\nA short, plump woman ...","The woman is of a petite stature, with a round...","[{'content': 'Describe the following: A short,..."
2,"Describe the following:\nA muscular, bald man ...","Describe the following:\nA muscular, bald man ...",The man before you exudes an aura of physical ...,[{'content': 'Describe the following: A muscul...
3,"Describe the following:\nA petite, elderly wom...","Describe the following:\nA petite, elderly wom...",The woman is small in stature and an advanced ...,[{'content': 'Describe the following: A petite...
4,"Describe the following:\nA lanky, teenage boy ...","Describe the following:\nA lanky, teenage boy ...","The boy is tall and slender, with long limbs t...","[{'content': 'Describe the following: A lanky,..."
...,...,...,...,...
48696,"Edit this sentence to be more descriptive: \""T...","Edit this sentence to be more descriptive: \""T...","The moist and fluffy cake had a divine, sweet ...",[{'content': 'Edit this sentence to be more de...
48697,Edit this paragraph to improve its coherence: ...,Edit this paragraph to improve its coherence: ...,I started my day early by going for a run befo...,[{'content': 'Edit this paragraph to improve i...
48698,"Edit this sentence to be more precise: \""I wan...","Edit this sentence to be more precise: \""I wan...",I desire to engage in the act of reading a book.,[{'content': 'Edit this sentence to be more pr...
48699,"Generate a list of 10 synonyms for \""sad\"".###...","Generate a list of 10 synonyms for \""sad\"".",1. Despondent\n2. Miserable\n3. Melancholy\n4....,[{'content': 'Generate a list of 10 synonyms f...


In [29]:
df['tok_len'] = df['messages'].apply(lambda x: len(tokenizer.apply_chat_template(x)))

Token indices sequence length is longer than the specified maximum sequence length for this model (3052 > 2048). Running this sequence through the model will result in indexing errors


In [30]:
df

,text,prompt,response,messages,tok_len
0,"Describe the following:\nA tall, thin man with...","Describe the following:\nA tall, thin man with...",The man is quite tall with a slender physique....,"[{'content': 'Describe the following: A tall, ...",145
1,"Describe the following:\nA short, plump woman ...","Describe the following:\nA short, plump woman ...","The woman is of a petite stature, with a round...","[{'content': 'Describe the following: A short,...",116
2,"Describe the following:\nA muscular, bald man ...","Describe the following:\nA muscular, bald man ...",The man before you exudes an aura of physical ...,[{'content': 'Describe the following: A muscul...,223
3,"Describe the following:\nA petite, elderly wom...","Describe the following:\nA petite, elderly wom...",The woman is small in stature and an advanced ...,[{'content': 'Describe the following: A petite...,86
4,"Describe the following:\nA lanky, teenage boy ...","Describe the following:\nA lanky, teenage boy ...","The boy is tall and slender, with long limbs t...","[{'content': 'Describe the following: A lanky,...",175
...,...,...,...,...,...
48696,"Edit this sentence to be more descriptive: \""T...","Edit this sentence to be more descriptive: \""T...","The moist and fluffy cake had a divine, sweet ...",[{'content': 'Edit this sentence to be more de...,44
48697,Edit this paragraph to improve its coherence: ...,Edit this paragraph to improve its coherence: ...,I started my day early by going for a run befo...,[{'content': 'Edit this paragraph to improve i...,102
48698,"Edit this sentence to be more precise: \""I wan...","Edit this sentence to be more precise: \""I wan...",I desire to engage in the act of reading a book.,[{'content': 'Edit this sentence to be more pr...,38
48699,"Generate a list of 10 synonyms for \""sad\"".###...","Generate a list of 10 synonyms for \""sad\"".",1. Despondent\n2. Miserable\n3. Melancholy\n4....,[{'content': 'Generate a list of 10 synonyms f...,79


In [31]:
df.describe()

,tok_len
count,48701.000000
mean,125.442640
std,133.498373
min,16.000000
25%,39.000000
50%,62.000000
75%,165.000000
max,3052.000000


In [32]:
df_1 = df[df['tok_len'] < 256].copy()

In [33]:
df_1.describe()

,tok_len
count,41476.000000
mean,79.249373
std,58.286086
min,16.000000
25%,37.000000
50%,52.000000
75%,110.000000
max,255.000000


In [34]:
len(df_1[df_1['tok_len'] < 128])

32775

In [35]:
#len(df_1[df_1['tok_len'] > 111])

In [36]:
df_2 = df_1[df_1['tok_len'] < 128]

In [37]:
df_2

,text,prompt,response,messages,tok_len
1,"Describe the following:\nA short, plump woman ...","Describe the following:\nA short, plump woman ...","The woman is of a petite stature, with a round...","[{'content': 'Describe the following: A short,...",116
3,"Describe the following:\nA petite, elderly wom...","Describe the following:\nA petite, elderly wom...",The woman is small in stature and an advanced ...,[{'content': 'Describe the following: A petite...,86
5,"Describe the following:\nA middle-aged, harrie...","Describe the following:\nA middle-aged, harrie...",The woman appears to be in her forties or fift...,[{'content': 'Describe the following: A middle...,126
8,Describe the following:\nA curvaceous woman wi...,Describe the following:\nA curvaceous woman wi...,"The woman is voluptuous, with curves that acce...",[{'content': 'Describe the following: A curvac...,100
11,"Describe the following:\nA flamboyant, extrove...","Describe the following:\nA flamboyant, extrove...","This man is loud and outgoing, with an outgoin...",[{'content': 'Describe the following: A flambo...,121
...,...,...,...,...,...
48694,"Edit this sentence to be more concise: \""I am ...","Edit this sentence to be more concise: \""I am ...","""I will buy milk from the store.""",[{'content': 'Edit this sentence to be more co...,41
48696,"Edit this sentence to be more descriptive: \""T...","Edit this sentence to be more descriptive: \""T...","The moist and fluffy cake had a divine, sweet ...",[{'content': 'Edit this sentence to be more de...,44
48697,Edit this paragraph to improve its coherence: ...,Edit this paragraph to improve its coherence: ...,I started my day early by going for a run befo...,[{'content': 'Edit this paragraph to improve i...,102
48698,"Edit this sentence to be more precise: \""I wan...","Edit this sentence to be more precise: \""I wan...",I desire to engage in the act of reading a book.,[{'content': 'Edit this sentence to be more pr...,38


In [38]:
df_3 = df_2[df_2['tok_len'] > 100]

In [39]:
df_3.describe()

,tok_len
count,2538.000000
mean,114.051615
std,7.914358
min,101.000000
25%,107.000000
50%,114.000000
75%,121.000000
max,127.000000


In [40]:
df_3[['messages']]

,messages
1,"[{'content': 'Describe the following: A short,..."
5,[{'content': 'Describe the following: A middle...
11,[{'content': 'Describe the following: A flambo...
12,[{'content': 'Describe the following: A stylis...
13,[{'content': 'Describe the following: A seriou...
...,...
48614,[{'content': 'Rearrange the given sentences in...
48646,[{'content': 'Edit this paragraph to improve i...
48659,[{'content': 'Edit this paragraph to improve i...
48692,[{'content': 'Edit this paragraph to improve i...


In [41]:
new_ds = Dataset.from_pandas(df_3[['messages']], preserve_index=False)

In [42]:
new_ds

Dataset({
    features: ['messages'],
    num_rows: 2538
})

In [43]:
#new_ds = new_ds.shuffle(seed=42).select(range(2500))

In [44]:
#new_ds

In [45]:
new_ds = new_ds.train_test_split(seed=42, test_size=0.1)
new_ds

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2284
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 254
    })
})

In [46]:
train_ds  = new_ds['train']
test_ds = new_ds['test']
train_ds, test_ds

(Dataset({
     features: ['messages'],
     num_rows: 2284
 }),
 Dataset({
     features: ['messages'],
     num_rows: 254
 }))

In [47]:
tokenizer

CodeGenTokenizerFast(name_or_path='microsoft/phi-2', vocab_size=50257, model_max_length=2048, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50257: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50258: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50259: AddedToken("                             ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50260: AddedToken("                            ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50261: AddedToken("           

In [48]:
model.config.pad_token_id

In [49]:
model.config.pad_token_id = tokenizer.pad_token_id

In [50]:
model.config.pad_token_id

50297

In [51]:
model.config.eos_token_id, tokenizer.eos_token_id

(50256, 50256)

In [52]:
tokenizer.padding_side='left'
tokenizer.padding_side

'left'

In [53]:
FORMAT_MAPPING['chatml'] == train_ds.features['messages']

True

In [54]:
FORMAT_MAPPING['chatml'] == test_ds.features['messages']

True

In [55]:
messages = train_ds["messages"][0]
print(len(messages))
output_texts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(output_texts)

2
<|user|>
Generate a list of 5 potential research topics for a sociology paper. (Input: None)<|end|>
<|assistant|>
1. The impact of social media on self-esteem and body image among teenagers
2. The role of race and ethnicity in shaping access to healthcare
3. The effects of income inequality on social mobility and economic participation
4. The influence of family structure on child development and academic achievement
5. The relationship between gender norms and domestic violence in intimate relationships.<|end|>
<|endoftext|>


In [56]:
messages = test_ds["messages"][10]
print(len(messages))
output_texts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(output_texts)

2
<|user|>
Generate a list of 5 potential interview questions for a job in marketing.<|end|>
<|assistant|>
1. How do you stay current on marketing trends and incorporate them into your strategies?
2. Can you describe a successful marketing campaign you have led in the past and the results it achieved?
3. How do you approach defining and targeting a specific audience for a product or service?
4. How do you measure the success and ROI of a marketing campaign?
5. Can you walk us through your process for developing a comprehensive marketing plan for a new product or service?<|end|>
<|endoftext|>


In [62]:
messages = test_ds["messages"][10]
print(len(messages))
output_texts = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
print(output_texts)

2
<|user|>
Generate a list of 5 potential interview questions for a job in marketing.<|end|>
<|assistant|>



In [58]:
messages = train_ds["messages"][0]
print(len(messages))
output_texts = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
print(output_texts)

2
<|user|>
Generate a list of 5 potential research topics for a sociology paper. (Input: None)<|end|>
<|assistant|>



# continue from here

In [63]:
# tokenize -> generate -> decode
max_length = 128
model_input = tokenizer(
      output_texts,
      truncation = True,
      max_length=max_length,
      padding="max_length",
      return_tensors='pt'
  ).to("cuda")


In [64]:
model.eval()

with torch.no_grad():
  output = model.generate(**model_input, max_new_tokens=256,
                          repetition_penalty=1.15,
                          pad_token_id=tokenizer.pad_token_id
                         )
  result = tokenizer.decode(output[0], skip_special_tokens=True)

  print(result)

<|user|>
Generate a list of 5 potential interview questions for a job in marketing.<|end|>
<|assistant|>
A: 1. Can you describe your experience with developing and executing successful marketing campaigns? 
2. How do you stay up-to-date on current trends and best practices in the industry? 
3. Have you worked with any specific software or tools to track and analyze data related to marketing efforts? 
4. What strategies have you used to increase brand awareness and customer engagement? 
5. Can you provide examples of how you have collaborated with other departments within the company to achieve marketing goals?



In [65]:
#it is trying to answer and is pretty good. Let us refine it further

In [88]:
sft_config = SFTConfig(
    ## GROUP 1: Memory usage
    # These arguments will squeeze the most out of your GPU's RAM
    # Checkpointing
    #gradient_checkpointing=True,
    # this saves a LOT of memory
    # Set this to avoid exceptions in newer versions of PyTorch
    #gradient_checkpointing_kwargs={'use_reentrant': False},
    # Gradient Accumulation / Batch size
    # Actual batch (for updating) is same (1x) as micro-batch size
    gradient_accumulation_steps=1,
    # The initial (micro) batch size to start off with
    per_device_train_batch_size=256,
    # If batch size would cause OOM, halves its size until it works
    auto_find_batch_size=True,

    ## GROUP 2: Dataset-related
    max_seq_length=128,
    # Dataset
    # packing a dataset means no padding is needed
    packing=False,

    ## GROUP 3: These are typical training parameters
    num_train_epochs=10,
    learning_rate=3e-4,
    # Optimizer
    # 8-bit Adam optimizer - doesn't help much if you're using LoRA!
    optim='adamw_torch',

    ## GROUP 4: Logging parameters
    logging_steps=100,
    logging_dir='./logs',
    output_dir='./phi3-chat-adapter',
    report_to='none',

    eval_strategy="epoch", # Evaluate the model every logging step
    #eval_steps=25,               # Evaluate and save checkpoints every 50 steps
    do_eval=True,                # Perform evaluation at the end of training
    save_strategy='epoch',
    load_best_model_at_end=True,
    )

In [89]:
tokenizer.padding_side='left'
#instruction_template = '<|user|>'
response_template = '<|assistant|>' # according to the tokenizer's chat template
collator_fn = DataCollatorForCompletionOnlyLM(response_template=response_template,
                                              tokenizer=tokenizer)

In [90]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator_fn
    )

Converting train dataset to ChatML:   0%|          | 0/2284 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/2284 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2284 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/254 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/254 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/254 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/254 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [91]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))
batch['input_ids'][0], batch['labels'][0]

(tensor([50297, 50297, 50297, 50297, 50297, 50297, 50297, 50297, 50297, 50297,
         50297, 50297, 50297, 50297, 50297, 50297, 50297, 50299,   198, 16594,
           257,  1430,   284, 15284,   262,  1989,   290, 25317,   286,   257,
         35991,  1813,   663,  9647,   290,  6001,    13, 50301,   198, 50300,
           198, 19457,    11,   994,   338,   262,  1430,    25,   198,   198,
         15506,    63,   198, 10394,   796,   642,    13,    21, 50286,     2,
          6330,   351,  2836,  5128,   198, 17015,   796,   807,    13,    17,
         50286,     2,  6330,   351,  2836,  5128,   198,   198, 20337,   796,
          9647,  1635,  6001,   198,   525, 16912,   796,   362,  1635,   357,
         10394,  1343,  6001,     8,   198,   198,  4798,  7203,   464,  1989,
           286,   262, 35991,   318,    25,  1600,  1989,     8,   198,  4798,
          7203,   464, 25317,   286,   262, 35991,   318,    25,  1600, 25317,
             8,   198, 15506,    63, 50301,   198, 5

In [92]:
len(batch['input_ids'][0])

127

In [93]:
trainer.train()

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Epoch,Training Loss,Validation Loss
1,No log,0.905177
2,0.855100,0.943419
3,0.717100,1.005128
4,0.717100,1.072543
5,0.564900,1.147461
6,0.429400,1.224497
7,0.347400,1.287628
8,0.347400,1.334423
9,0.280800,1.376293
10,0.248300,1.400806


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explic

TrainOutput(global_step=720, training_loss=0.48479072716501026, metrics={'train_runtime': 8608.7269, 'train_samples_per_second': 2.653, 'train_steps_per_second': 0.084, 'total_flos': 4.6410746078208e+16, 'train_loss': 0.48479072716501026})

In [94]:
torch.cuda.empty_cache()

In [113]:
def gen_output(messages):

  output_texts = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
  print(output_texts)

  # tokenize -> generate -> decode
  max_length = 128
  model_input = tokenizer(
      output_texts,
      truncation = True,
      max_length=max_length,
      padding = "max_length",
      return_tensors='pt'
  ).to("cuda")

  model.eval()
  with torch.no_grad():
    output = model.generate(**model_input, max_new_tokens=128,
                            repetition_penalty=1.5,
                            eos_token_id=tokenizer.eos_token_id,
                            pad_token_id=tokenizer.pad_token_id,
                            num_beams=4,
                            top_k=3,
                            do_sample=True,
                            early_stopping=True
                            )
  result = tokenizer.decode(output[0], skip_special_tokens=True)
  print(result)
  return result

In [108]:
!pip install sacrebleu evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 186.4 MB/s eta 0:00:00


In [120]:
def gen_out(i):
    messages = test_ds["messages"][i]
    result = gen_output(messages)
    print("###############################")
    print(f'Original output\n {tokenizer.apply_chat_template(messages, tokenize=False)}')
    orig_out = tokenizer.apply_chat_template(messages, tokenize=False)
    print("###############################")

In [135]:
indx = [5, 15, 25, 35, 45, 55, 105, 205, 9]
indx, len(indx)

([5, 15, 25, 35, 45, 55, 105, 205, 9], 9)

In [122]:
gen_out(indx[0])

<|user|>
Create a list of 5 famous artists and their works.<|end|>
<|assistant|>

<|user|>
Create a list of 5 famous artists and their works.<|end|>
<|assistant|>
1. Leonardo da Vinci - Mona Lisa, The Last Supper, Vitruvian Man
2. Vincent van Gogh - Starry Night, Sunflowers, The Potato Eaters
3. Pablo Picasso - Guernica, Les Demoiselles d'Avignon, The Weeping Woman
4. Frida Kahlo - The Two Fridas, Self-Portrait with Thorn Necklace and Hummingbird, The Broken Column
5. Claude Monet - Water Lilies, Impression, Sunrise, A Sunday on La Grande Jatte=~=~

###############################
Original output
 <|user|>
Create a list of 5 famous artists and their works.<|end|>
<|assistant|>
1. Leonardo da Vinci - Mona Lisa, The Last Supper 
2. Vincent van Gogh - The Starry Night, Sunflowers
3. Pablo Picasso - Les Demoiselles d'Avignon, Guernica 
4. Michelangelo - Sistine Chapel ceiling, David 
5. Claude Monet - Water Lilies, Impression, Sunrise<|end|>
<|endoftext|>
###############################


In [123]:
gen_out(indx[1])

<|user|>
Research and compile a list of the best sustainable clothing brands.<|end|>
<|assistant|>

<|user|>
Research and compile a list of the best sustainable clothing brands.<|end|>
<|assistant|>
1. Patagonia 
2. Everlane 
3. Reformation 
4. Allbirds 
5. The North Face 
6. Stella McCartney 
7. Eileen Fisher 
8. Girlfriend Collective 
9. Rothy's 
10. Tencel Threads 
11. People Tree 
12. H&M Conscious Collection 
13. Zara Conscious Collection 
14. Levi Strauss & Co. 
15. Adidas 
16. Nike 
17. Under Armour 
18. Puma 
19. Reebok 
###############################
Original output
 <|user|>
Research and compile a list of the best sustainable clothing brands.<|end|>
<|assistant|>
1. Patagonia 
2. Levi's 
3. Reformation 
4. Everlane 
5. People Tree 
6. Eileen Fisher 
7. Pact 
8. Veja 
9. Kotn 
10. Amour Vert 
11. Alternative Apparel 
12. Nudie Jeans 
13. Outerknown 
14. Rothy's 
15. Thought Clothing.<|end|>
<|endoftext|>
###############################


In [124]:
gen_out(indx[2])

<|user|>
Classify these flowers based on their color: rose, daisy, lily, sunflower, and orchid.<|end|>
<|assistant|>

<|user|>
Classify these flowers based on their color: rose, daisy, lily, sunflower, and orchid.<|end|>
<|assistant|>
rose - red, pink, white, yellow
daisy - white, yellow
lily - white, pink, purple, yellow
sunflower - yellow
orchid - white, pink, purple, yellow, red, blue, green, brown, black

###############################
Original output
 <|user|>
Classify these flowers based on their color: rose, daisy, lily, sunflower, and orchid.<|end|>
<|assistant|>
- Rose: commonly found in shades of red, pink, white, and yellow
- Daisy: typically white with a yellow center, but can also be pink or purple
- Lily: commonly found in shades of white, pink, yellow, and orange
- Sunflower: bright yellow with a dark center
- Orchid: can come in a variety of colors such as white, pink, purple, yellow, and orange<|end|>
<|endoftext|>
###############################


In [125]:
gen_out(indx[3])

<|user|>
Write a brief description of a place you visited that made you feel happy.<|end|>
<|assistant|>

<|user|>
Write a brief description of a place you visited that made you feel happy.<|end|>
<|assistant|>
The place I visited that made me feel happy was a small town in the countryside. The air was fresh and the scenery was breathtaking, with rolling hills and fields of wildflowers. The locals were friendly and welcoming, and I felt a sense of peace and contentment as I explored the quaint shops and cafes. The sound of birds chirping and the smell of freshly baked bread filled the air, creating a warm and inviting atmosphere. Overall, it was a perfect escape from the hustle and bustle of city life, and I left feeling rejuvenated and grateful for the beauty of nature and the kindness of strangers. guiName

###############################
Original output
 <|user|>
Write a brief description of a place you visited that made you feel happy.<|end|>
<|assistant|>
I recently visited a secl

In [126]:
gen_out(indx[3])

<|user|>
Write a brief description of a place you visited that made you feel happy.<|end|>
<|assistant|>

<|user|>
Write a brief description of a place you visited that made you feel happy.<|end|>
<|assistant|>
I visited the beach in Hawaii and it made me feel incredibly happy. The sun was shining, the water was crystal clear, and the sand was soft and warm beneath my toes. I spent the day lounging on the beach, reading a good book, and taking long walks along the shore. The sound of the waves crashing against the shore and the salty sea air filled me with a sense of peace and contentment. It was the perfect getaway from the hustle and bustle of everyday life and I left feeling rejuvenated and grateful for the opportunity to experience such a beautiful place. guiName

###############################
Original output
 <|user|>
Write a brief description of a place you visited that made you feel happy.<|end|>
<|assistant|>
I recently visited a secluded beach on the coast of Maui, Hawaii. T

In [128]:
gen_out(indx[4])

<|user|>
Using R programming language, create a decision tree to predict whether a customer will purchase a product. (Input: customer data such as age, gender, income, and purchase history)<|end|>
<|assistant|>

<|user|>
Using R programming language, create a decision tree to predict whether a customer will purchase a product. (Input: customer data such as age, gender, income, and purchase history)<|end|>
<|assistant|>
Sure, I can create a decision tree using R programming language to predict whether a customer will purchase a product based on their age, gender, income, and purchase history. 

First, we need to import the necessary libraries such as dplyr, tidyr, and caret. Then, we can load the customer data into a data frame and preprocess it by removing any missing values or outliers. 

Next, we can split the data into training and testing sets using the train_test_split function from the caret library. We can then use the RandomForestClassifier function from the caret library
#####

In [129]:
gen_out(indx[5])

<|user|>
In 3-5 sentences, explain why recycling is important for the environment.<|end|>
<|assistant|>

<|user|>
In 3-5 sentences, explain why recycling is important for the environment.<|end|>
<|assistant|>
Recycling is important for the environment because it reduces the amount of waste that ends up in landfills and incinerators, which can release harmful pollutants into the air and water. Recycling also conserves natural resources by reducing the need for new materials to be extracted from the earth. Additionally, recycling helps to reduce greenhouse gas emissions associated with the production of new materials, which can contribute to climate change. Overall, recycling is an important part of sustainable living and helps to protect the health of our planet for future generations.=~=~

###############################
Original output
 <|user|>
In 3-5 sentences, explain why recycling is important for the environment.<|end|>
<|assistant|>
Recycling reduces the amount of waste that goe

In [130]:
gen_out(indx[6])

<|user|>
Classify a set of 10 vehicles based on their fuel efficiency.<|end|>
<|assistant|>

<|user|>
Classify a set of 10 vehicles based on their fuel efficiency.<|end|>
<|assistant|>
1. Tesla Model S - 100 miles per gallon
2. Toyota Prius - 50 miles per gallon
3. Honda Civic - 40 miles per gallon
4. Ford F-150 - 25 miles per gallon
5. Chevrolet Camaro - 20 miles per gallon
6. BMW i3 - 60 miles per gallon
7. Volkswagen Golf - 30 miles per gallon
8. Audi A4 - 35 miles per gallon
9. Nissan Leaf - 100 miles per gallon
10. Hyundai Ioniq - 40 miles per gallon trademore

###############################
Original output
 <|user|>
Classify a set of 10 vehicles based on their fuel efficiency.<|end|>
<|assistant|>
1. Toyota Prius - Hybrid/Electric
2. Tesla Model S - Electric
3. Honda Fit - Gasoline/electric hybrid
4. Nissan Leaf - Electric
5. Mitsubishi Mirage - Gasoline
6. BMW i3 - Electric
7. Chevrolet Bolt - Electric
8. Kia Niro - Gasoline/electric hybrid
9. Toyota Corolla - Gasoline
10. Ford

In [131]:
gen_out(indx[7])

<|user|>
Provide 3 arguments for why television is still relevant today.<|end|>
<|assistant|>

<|user|>
Provide 3 arguments for why television is still relevant today.<|end|>
<|assistant|>
1. Television remains a primary source of news and information for millions of people around the world. With the rise of 24-hour news channels and online news outlets, television continues to play a crucial role in keeping people informed about current events.

2. Television also serves as a platform for entertainment and cultural expression. From sitcoms and dramas to reality shows and documentaries, television offers a wide range of programming that appeals to diverse audiences and reflects the values and interests of different communities.

3. Finally, television has evolved to incorporate new technologies and platforms, such as streaming services and social media, which have expanded its reach and relevance
###############################
Original output
 <|user|>
Provide 3 arguments for why tele

In [141]:
trainer.save_model('phi_2_instruct_hf-chat-guanaco')

In [138]:
from huggingface_hub import login
login()

In [140]:
trainer.push_to_hub()

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.56k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/94.4M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Fardan/phi3-chat-adapter/commit/1dde0f40d8e7fc406b585de75499591a30db7cb0', commit_message='End of training', commit_description='', oid='1dde0f40d8e7fc406b585de75499591a30db7cb0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Fardan/phi3-chat-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='Fardan/phi3-chat-adapter'), pr_revision=None, pr_num=None)